In [1]:
import sys

import matplotlib
import numpy
import pandas
import sklearn

print(sys.version.split()[0], numpy.__version__, pandas.__version__, sklearn.__version__, matplotlib.__version__)

3.12.8 2.3.4 2.3.3 1.7.2 3.10.7


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split

SEED = 2175
data = pd.read_csv("data/spambase.csv")
feature_names = [c for c in data.columns if c != "spam"]
X, y = data[feature_names].to_numpy(), data["spam"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print(X.shape, round(100 * y.mean(), 1), X_train.shape, X_test.shape)

assert X.shape == (4204, 57)
assert X_train.shape == (3153, 57) and X_test.shape == (1051, 57)
assert not data.isna().any().any()
assert set(np.unique(y)) == {0, 1}

(4204, 57) 39.9 (3153, 57) (1051, 57)


## Task 4: Random Forests & Ensembles

MOP 3231, Machine Learning (Python)

SEED = 2175 (last four digits of my student ID).

The positive class is spam. A false positive is a legitimate e-mail sent to spam, so the filter's target is at most 1% false positives. This is an old teaching dataset, so these results do not describe today's spam filters.

In [3]:
from sklearn.ensemble import AdaBoostClassifier, BaggingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(random_state=SEED)
bag = BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, oob_score=True, random_state=SEED)
rf = RandomForestClassifier(n_estimators=300, oob_score=True, random_state=SEED, n_jobs=-1)
lr = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(), LogisticRegression(max_iter=2000))
bag_lr = BaggingClassifier(lr, n_estimators=50, random_state=SEED)
knn = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(), KNeighborsClassifier(n_neighbors=15))
ada = AdaBoostClassifier(n_estimators=400, random_state=SEED)
members = [("lr", lr), ("knn", knn), ("tree", tree)]

All comparisons use the same five training folds. I report accuracy unless another metric is named, and standard deviations use `ddof=0`. A paired difference is inside the fold spread when `abs(mean difference) <= std of differences`; in that case I do not name a winner. The test split reports results and does not choose thresholds, members or numbers of rounds.

In [4]:
import math
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_validate

def score_table(results):
    """Show five scores or paired differences, followed by their mean and std."""
    return pd.DataFrame(
        {name: list(values) + [values.mean(), values.std()] for name, values in results.items()},
        index=["fold 1", "fold 2", "fold 3", "fold 4", "fold 5", "mean", "std"],
    ).T

cv_scores = {}

### Part 1 · Bagging and out-of-bag estimates

In [5]:
def oob_mask(sample_indices, n_samples):
    """Return a Boolean mask of rows absent from a bootstrap sample."""
    mask = np.ones(n_samples, dtype=bool)
    mask[np.asarray(sample_indices, dtype=int)] = False
    return mask

In [6]:
assert np.array_equal(oob_mask([0, 2, 2], 5), [False, True, False, True, True])
assert np.array_equal(oob_mask([], 3), [True, True, True])
indices = np.array([3, 0])
saved_indices = indices.copy()
mask = oob_mask(sample_indices=indices, n_samples=4)
assert np.array_equal(mask, [False, True, True, False])
assert mask.dtype == np.bool_ and mask.shape == (4,)
assert np.array_equal(indices, saved_indices)
assert oob_mask([], 0).shape == (0,)
print("oob_mask checks passed")

oob_mask checks passed


A bootstrap sample can contain the same row several times. I first count which rows each tree did not see, then let only those trees vote for a row. A tied vote goes to class 0.

In [7]:
bag.fit(X_train, y_train)
n_train = len(y_train)
oob_votes = np.zeros((n_train, 2), dtype=int)
oob_shares = []

for estimator, sample_indices, columns in zip(
        bag.estimators_, bag.estimators_samples_, bag.estimators_features_):
    mask = oob_mask(sample_indices, n_train)
    oob_shares.append(mask.mean())
    predictions = estimator.predict(X_train[mask][:, columns]).astype(int)
    oob_votes[np.flatnonzero(mask), predictions] += 1

expected_oob = (1 - 1 / n_train) ** n_train
display(pd.DataFrame({
    "observed": [np.mean(oob_shares), 1 - np.mean(oob_shares)],
    "formula": [expected_oob, 1 - expected_oob],
}, index=["out of bag", "in bag"]).round(6))

assert (oob_votes.sum(axis=1) > 0).all()
manual_oob = accuracy_score(y_train, oob_votes.argmax(axis=1))
print(f"manual OOB accuracy: {manual_oob:.6f}")
print(f"sklearn OOB accuracy: {bag.oob_score_:.6f}")
assert np.isclose(manual_oob, bag.oob_score_)

,observed,formula
out of bag,0.367507,0.367821
in bag,0.632493,0.632179


manual OOB accuracy: 0.936568
sklearn OOB accuracy: 0.936568


In [8]:
for name, model in {"tree": tree, "bag": bag, "rf": rf, "lr": lr, "bag_lr": bag_lr}.items():
    cv_scores[name] = cross_validate(
        model, X_train, y_train, cv=cv, scoring="accuracy")["test_score"]

display(score_table(cv_scores).round(4))
part1_differences = {
    "bag - tree": cv_scores["bag"] - cv_scores["tree"],
    "rf - tree": cv_scores["rf"] - cv_scores["tree"],
    "bag_lr - lr": cv_scores["bag_lr"] - cv_scores["lr"],
}
display(score_table(part1_differences).round(4))

,fold 1,fold 2,fold 3,fold 4,fold 5,mean,std
tree,0.9002,0.8796,0.8859,0.8984,0.9032,0.8934,0.0091
bag,0.9334,0.9176,0.9160,0.9222,0.9476,0.9274,0.0118
rf,0.9414,0.9303,0.9319,0.9349,0.9460,0.9369,0.0059
lr,0.9445,0.9239,0.9303,0.9159,0.9397,0.9309,0.0104
bag_lr,0.9445,0.9271,0.9319,0.9175,0.9413,0.9324,0.0098


,fold 1,fold 2,fold 3,fold 4,fold 5,mean,std
bag - tree,0.0333,0.0380,0.0301,0.0238,0.0444,0.0339,0.0070
rf - tree,0.0412,0.0507,0.0460,0.0365,0.0429,0.0434,0.0047
bag_lr - lr,0.0000,0.0032,0.0016,0.0016,0.0016,0.0016,0.0010


In [9]:
rf.fit(X_train, y_train)
rf_test_accuracy = rf.score(X_test, y_test)
rf_test_se = np.sqrt(rf_test_accuracy * (1 - rf_test_accuracy) / len(y_test))
rf_cv_mean = cv_scores["rf"].mean()
rf_cv_std = cv_scores["rf"].std()

print(f"forest OOB accuracy: {rf.oob_score_:.4f}")
print(f"forest CV accuracy: {rf_cv_mean:.4f} +/- {rf_cv_std:.4f}")
print(f"forest test accuracy: {rf_test_accuracy:.4f}; SE: {rf_test_se:.4f}")
print(f"absolute OOB-CV gap: {abs(rf.oob_score_ - rf_cv_mean):.4f}")
print("OOB and CV agree by the handout rule:", abs(rf.oob_score_ - rf_cv_mean) <= rf_cv_std)
print(f"OOB-test gap: {abs(rf.oob_score_ - rf_test_accuracy):.4f}")
print(f"CV-test gap: {abs(rf_cv_mean - rf_test_accuracy):.4f}")

forest OOB accuracy: 0.9445
forest CV accuracy: 0.9369 +/- 0.0059
forest test accuracy: 0.9572; SE: 0.0062
absolute OOB-CV gap: 0.0076
OOB and CV agree by the handout rule: False
OOB-test gap: 0.0127
CV-test gap: 0.0203


Reading the results:

- The mean OOB share is 0.367507 against the formula's 0.367821; the in-bag share is 0.632493 against 0.632179. Sampling 3,153 times does not give 3,153 different rows. My OOB accuracy is 0.936568, the same as scikit-learn's.
- Bagging improves the tree by 0.0339, with a paired std of 0.0070. The forest improves on the tree by 0.0434, with std 0.0047. Both gains are outside the paired spread. A deep tree changes a lot when its training rows change; averaging many such trees reduces that variance. Random feature choices also make forest trees less alike.
- Bagging logistic regression improves accuracy by only 0.0016, with paired std 0.0010. Under the handout's rule this is a small improvement outside the spread, rather than a tie. Logistic regression is already a stable learner, so bootstrap averaging has much less variance to remove than it does for a deep tree.
- Forest OOB accuracy is 0.9445, while CV gives 0.9369 with std 0.0059. Their gap of 0.0076 is larger than that std, so OOB is not a fair stand-in by the stated rule on this split. Test accuracy is higher, at 0.9572 with SE 0.0062; its gaps from OOB (0.0127) and CV (0.0203) are both larger than one test SE. The three estimates use different training/evaluation rows, and the test set appears easier here; I do not claim that the three numbers agree.